# Medallón · Análisis y ML de finanzas personales

Este notebook lee las capas **Plata** y **Oro** que exporta la app y entrena modelos sobre ellas.

```
data/
├── bronce/  medallon-respaldo.json
├── plata/   plata_movimientos.csv
└── oro/     oro_mensual.csv · oro_semanal.csv
```

1. Exploración de la capa Plata
2. Pronóstico de gastos e ingresos (Ridge con rezagos, validado en el tiempo)
3. Detección de gastos inusuales (Isolation Forest)
4. Clasificador de categoría a partir de la nota
5. Perfiles de semanas (K-Means)
6. Guardado de resultados de vuelta en Oro

Requisitos: `pip install pandas numpy scikit-learn matplotlib`

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.cluster import KMeans

plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False, "axes.spines.right": False})
DATA = next((p for p in (Path("data"), Path("../data")) if p.exists()), Path("data"))   # cambia la ruta si guardaste los CSV en otro lugar

plata = pd.read_csv(DATA/"plata"/"plata_movimientos.csv", parse_dates=["fecha", "semana"])
oro_s = pd.read_csv(DATA/"oro"/"oro_semanal.csv", parse_dates=["periodo"])
oro_m = pd.read_csv(DATA/"oro"/"oro_mensual.csv")
fmt = lambda v: f"${v:,.0f}"
print(f"Plata: {len(plata)} filas · Oro: {len(oro_m)} meses, {len(oro_s)} semanas")
plata.head()

## 1. Exploración (Plata)

In [ ]:
gastos = plata[plata.tipo == "gasto"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
gastos.groupby("categoria").monto.sum().sort_values().plot.barh(ax=ax[0], color="#0a84ff", title="Gasto total por categoría")
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
gastos.groupby("dia_semana").monto.sum().reindex(range(7), fill_value=0).set_axis(dias).plot.bar(ax=ax[1], color="#30d158", title="Gasto por día de la semana", rot=0)
for a in ax: a.set_xlabel(""); a.set_ylabel("")
plt.tight_layout(); plt.show()

## 2. Pronóstico

Los gastos fijos se conocen de antemano, así que el modelo predice solo la parte **variable** y le suma el fijo promedio. Se usan rezagos (`lag1…lag4`) y la media móvil de 4 periodos. La validación es **walk-forward**: se entrena con el pasado y se predice el siguiente periodo, sin mirar el futuro. El error se compara con una línea base ingenua ("repetir el periodo anterior"); si el modelo no la supera, conviene confiar poco en él.

In [ ]:
def semanas_completas(df, col):
    s = df.set_index("periodo")[col].sort_index()
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="7D"), fill_value=0)
    return s[s.index + pd.Timedelta(days=6) < pd.Timestamp.today().normalize()]

def con_rezagos(y, n):
    d = pd.DataFrame({"y": y})
    for k in range(1, n + 1): d[f"lag{k}"] = y.shift(k)
    d["media"] = y.shift(1).rolling(min(n, 4)).mean()
    return d.dropna()

def pronosticar(y, n=4, min_train=8):
    d = con_rezagos(y, n)
    if len(d) <= min_train:
        return None
    X, t = d.drop(columns="y"), d["y"]
    err, base = [], []
    for i in range(min_train, len(d)):
        m = Ridge(alpha=1.0).fit(X.iloc[:i], t.iloc[:i])
        err.append(abs(m.predict(X.iloc[[i]])[0] - t.iloc[i]))
        base.append(abs(X["lag1"].iloc[i] - t.iloc[i]))
    m = Ridge(alpha=1.0).fit(X, t)
    ult = pd.DataFrame([{**{f"lag{k}": y.iloc[-k] for k in range(1, n + 1)}, "media": y.iloc[-min(n, 4):].mean()}])[X.columns]
    return dict(pred=max(m.predict(ult)[0], 0), mae=np.mean(err), mae_base=np.mean(base))

# Los gastos fijos son predecibles (se conocen de antemano), así que el modelo aprende solo lo variable
vari = plata[(plata.tipo == "gasto") & (plata.origen == "manual")].groupby("semana").monto.sum()
vari = vari.reindex(pd.date_range(plata.semana.min(), plata.semana.max(), freq="7D"), fill_value=0)
vari = vari[vari.index + pd.Timedelta(days=6) < pd.Timestamp.today().normalize()]
fijo_sem = plata[(plata.tipo == "gasto") & (plata.origen == "fijo")].groupby("mes").monto.sum().mean() * 12 / 52
fijo_sem = 0 if np.isnan(fijo_sem) else fijo_sem

resultados = []
for nombre, y, n, mt, extra in [
    ("Gastos · próxima semana", vari, 4, 8, fijo_sem),
    ("Gastos · próximo mes",    oro_m.set_index("periodo").gastos,   3, 3, 0),
    ("Ingresos · próximo mes",  oro_m.set_index("periodo").ingresos, 3, 3, 0),
]:
    r = pronosticar(y, n, mt)
    if r is None:
        print(f"{nombre}: datos insuficientes (necesitas más historial)"); continue
    mejora = 1 - r["mae"] / r["mae_base"] if r["mae_base"] else np.nan
    total = r["pred"] + extra
    print(f"{nombre}: ≈ {fmt(total)}" + (f" (variable {fmt(r['pred'])} + fijos {fmt(extra)})" if extra else "")
          + f" · error medio {fmt(r['mae'])} vs línea base {fmt(r['mae_base'])} ({mejora:+.0%})")
    resultados.append({"serie": nombre, "pronostico": round(total), "mae": round(r["mae"]), "mae_base": round(r["mae_base"])})
pron = pd.DataFrame(resultados); pron

In [ ]:
y = semanas_completas(oro_s, "gastos")
ax = y.tail(26).plot(color="#0a84ff", marker="o", title="Gasto semanal (últimas 26 semanas)")
ax.axhline(y.tail(26).mean(), color="gray", ls="--", lw=1); ax.set_xlabel(""); plt.show()

## 3. Gastos inusuales (Isolation Forest)

Compara cada gasto manual con lo normal de su categoría. Los fijos se excluyen porque se repiten por diseño.

In [ ]:
g = plata[(plata.tipo == "gasto") & (plata.origen == "manual")].copy()
g["ratio_cat"] = g.monto / g.groupby("categoria").monto.transform("median")
g["log_monto"] = np.log1p(g.monto)
if len(g) >= 30:
    X = g[["log_monto", "ratio_cat", "dia_semana"]]
    g["anomalo"] = IsolationForest(contamination=0.03, random_state=0).fit_predict(X) == -1
    anom = g[g.anomalo].sort_values("monto", ascending=False)[["fecha", "categoria", "nota", "monto", "ratio_cat"]]
    anom["ratio_cat"] = anom.ratio_cat.round(1)
    print(f"{len(anom)} gastos marcados como inusuales de {len(g)}")
else:
    anom = pd.DataFrame(); print("Se necesitan al menos 30 gastos manuales")
anom

## 4. Clasificador de categoría

Aprende de tus notas (por ejemplo "Mercado" → Comida) para sugerir la categoría de nuevos gastos. Usa n-gramas de caracteres, así tolera errores de escritura.

In [ ]:
c = plata[(plata.tipo == "gasto") & plata.nota.notna() & (plata.origen == "manual")].copy()
c["nota"] = c.nota.astype(str).str.lower().str.strip()
c = c[c.groupby("categoria").categoria.transform("size") >= 3]
if len(c) >= 30 and c.categoria.nunique() >= 2:
    clf = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)), LogisticRegression(max_iter=1000))
    acc = cross_val_score(clf, c.nota, c.categoria, cv=3).mean()
    clf.fit(c.nota, c.categoria)
    print(f"Exactitud (validación cruzada): {acc:.0%}")
    for t in ["supermercado", "uber", "farmacia", "cine"]:
        print(f"  {t!r:16} → {clf.predict([t])[0]}")
else:
    clf = None; print("Se necesitan ≥30 gastos con nota y categorías con ≥3 ejemplos")

## 5. Perfiles de semanas (K-Means)

Agrupa las semanas según cómo repartes el gasto entre categorías (proporciones, no montos).

In [ ]:
pv = plata[plata.tipo == "gasto"].pivot_table(index="semana", columns="categoria", values="monto", aggfunc="sum", fill_value=0)
if len(pv) >= 9:
    share = pv.div(pv.sum(axis=1), axis=0)
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(share)
    pv["perfil"] = km.labels_
    resumen = pd.DataFrame(km.cluster_centers_, columns=share.columns).round(2)
    resumen["semanas"] = np.bincount(km.labels_)
    resumen["gasto_medio"] = pv.groupby("perfil").sum(numeric_only=True).sum(axis=1).div(resumen.semanas).round(0)
    display(resumen)
else:
    print("Se necesitan al menos 9 semanas con gastos")

## 6. Guardar resultados en Oro

Los resultados vuelven a la capa Oro para que el siguiente análisis (o un dashboard) los reutilice.

In [ ]:
out = DATA/"oro"; out.mkdir(parents=True, exist_ok=True)
pron.to_csv(out/"pronostico.csv", index=False)
if len(anom): anom.to_csv(out/"gastos_inusuales.csv", index=False)
print("Guardado en", out.resolve())